In [56]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, to_date, regexp_replace,trim, upper, when, create_map, lit, current_date, to_date, last_day, concat, count, sum,avg
from pyspark.sql.types import DecimalType, StringType

spark=SparkSession.builder\
.appName('data engineering project')\
.getOrCreate()

In [16]:
transactions=spark.read.csv('data/transactions_data.csv',
                           header=True,
                           inferSchema=True,
                           sep=',',
                           quote='"',
                           escape='"',
                           multiLine=True,
                           mode='PERMISSIVE')
transactions.printSchema()
transactions.describe()

[Stage 23:>                                                         (0 + 1) / 1]

root
 |-- id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- amount: string (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- errors: string (nullable = true)



DataFrame[summary: string, id: string, client_id: string, card_id: string, amount: string, use_chip: string, merchant_id: string, merchant_city: string, merchant_state: string, zip: string, mcc: string, errors: string]

In [17]:
users=spark.read.csv('data/users_data.csv',
                     header=True,
                     inferSchema=True,
                     sep=',',
                     quote='"',
                     escape='"',
                     multiLine=True,
                     mode='PERMISSIVE')
users.printSchema()

root
 |-- id: integer (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: string (nullable = true)
 |-- yearly_income: string (nullable = true)
 |-- total_debt: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)



In [18]:
cards=spark.read.csv('data/cards_data.csv',
                     header=True,
                     inferSchema=True,
                     sep=',',
                     quote='"',
                     escape='"',
                     multiLine=True,
                     mode='PERMISSIVE')
cards.printSchema()

root
 |-- id: integer (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: integer (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- credit_limit: string (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: integer (nullable = true)
 |-- card_on_dark_web: string (nullable = true)



In [19]:
transactions.filter(transactions.id.isNotNull())\
    .select('id')\
    .show(truncate=False)

+-------+
|id     |
+-------+
|7475327|
|7475328|
|7475329|
|7475331|
|7475332|
|7475333|
|7475334|
|7475335|
|7475336|
|7475337|
|7475338|
|7475339|
|7475340|
|7475341|
|7475342|
|7475343|
|7475344|
|7475345|
|7475346|
|7475347|
+-------+
only showing top 20 rows


In [20]:
transactions_processed=transactions\
    .withColumnRenamed('id','transaction_id')\
    .withColumn('amount', regexp_replace('amount', '[^0-9.]','').cast(DecimalType(12,2)))\
    .withColumn('transaction_date', to_date('date'))\
    .withColumn('zip_code',col('zip').cast('string'))\
    .withColumn(
        'has_error',
        when(col('errors').isNotNull(), True).otherwise(False))\
    .withColumn('merchant_city', upper(trim(col('merchant_city'))))\
    .withColumn('merchant_state', upper(trim(col('merchant_state'))))\
    .withColumn(
        'use_chip',
        when(upper(col('use_chip')) == 'ONLINE TRANSACTION', 'Yes')
        .when(upper(col('use_chip')) == 'SWIPE TRANSACTION', 'Yes')
        .when(upper(col('use_chip')) == 'NULL', 'No')
        .otherwise(None))\
    .fillna({
    'zip_code': 'ONLINE',
    'merchant_state': 'ONLINE'})\
    .filter(col('transaction_id').isNotNull())\
    .filter(col('client_id').isNotNull())\
    .filter(col('amount') > 0)\
    .drop('date')\
    .drop('errors')\
    .drop('zip')\
    .dropDuplicates(['transaction_id'])

In [21]:
transactions_processed.show(truncate=False)

[Stage 29:>                                                         (0 + 1) / 1]

+--------------+---------+-------+------+--------+-----------+-------------+--------------+----+----------------+--------+---------+
|transaction_id|client_id|card_id|amount|use_chip|merchant_id|merchant_city|merchant_state|mcc |transaction_date|zip_code|has_error|
+--------------+---------+-------+------+--------+-----------+-------------+--------------+----+----------------+--------+---------+
|7475336       |335      |5131   |261.58|Yes     |50292      |ONLINE       |ONLINE        |7801|2010-01-01      |ONLINE  |false    |
|7475338       |554      |3912   |3.51  |Yes     |67570      |PEARLAND     |TX            |5311|2010-01-01      |77581.0 |false    |
|7475353       |301      |3742   |10.17 |Yes     |39021      |ONLINE       |ONLINE        |4784|2010-01-01      |ONLINE  |false    |
|7475356       |566      |3439   |16.86 |Yes     |16798      |ONLINE       |ONLINE        |4121|2010-01-01      |ONLINE  |false    |
|7475365       |820      |127    |270.22|Yes     |73186      |ONLINE 

In [22]:
month_map = {
    1: "JAN", 2: "FEB", 3: "MAR", 4: "APR",
    5: "MAY", 6: "JUN", 7: "JUL", 8: "AUG",
    9: "SEP", 10: "OCT", 11: "NOV", 12: "DEC"}

mapping_expr = create_map(
    *[lit(x) for pair in month_map.items() for x in pair])

users_processed= users\
    .withColumnRenamed('id', 'client_id')\
    .withColumn('birth_month_name', mapping_expr[col('birth_month')])\
    .withColumn('yearly_income', regexp_replace(col('yearly_income'), '[^0-9.]', '').cast(DecimalType(12,2)))\
    .withColumn('per_capita_income', regexp_replace(col('per_capita_income'), '[^0-9.]', '').cast(DecimalType(12,2)))\
    .withColumn('total_debt', regexp_replace(col('total_debt'), '[^0-9.]', '').cast(DecimalType(12,2)))\
    .filter((col('credit_score') >= 300) & (col('credit_score') <=850))\
    .filter((col('current_age') >0) & (col('current_age')<=100))\
    .filter(col('client_id').isNotNull())\
    .dropDuplicates(['client_id'])

In [23]:
users_processed.show(truncate=False)

+---------+-----------+--------------+----------+-----------+------+-----------------------------+--------+---------+-----------------+-------------+----------+------------+----------------+----------------+
|client_id|current_age|retirement_age|birth_year|birth_month|gender|address                      |latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|birth_month_name|
+---------+-----------+--------------+----------+-----------+------+-----------------------------+--------+---------+-----------------+-------------+----------+------------+----------------+----------------+
|0        |33         |69            |1986      |3          |Male  |858 Plum Avenue              |43.59   |-70.33   |29237.00         |59613.00     |36199.00  |763         |4               |MAR             |
|1        |43         |74            |1976      |4          |Female|113 Burns Lane               |30.44   |-87.18   |22247.00         |45360.00     |14587.00  |704     

In [24]:
cards_processed=cards\
    .withColumnRenamed('id', 'card_id')\
    .withColumn('credit_limit', regexp_replace(col('credit_limit'), '[^0-9.]', '').cast(DecimalType(12,2)))\
    .withColumn('expires', last_day(to_date(col('expires'), 'MM/yyyy')))\
    .withColumn('acct_open_date', to_date(concat(lit('01/'), col('acct_open_date')),'dd/MM/yyyy'))\
    .withColumn(
        'is_expired',
        when(col('expires') < current_date(), True).otherwise(False))\
    .withColumn('card_brand', upper(trim(col('card_brand'))))\
    .withColumn('card_type', upper(trim(col('card_type'))))\
    .withColumn(
        'has_chip',
        when(upper(col('has_chip')) == 'YES', True)
        .when(upper(col('has_chip')) == 'NO', False)
        .otherwise(None))\
    .withColumn(
        'card_on_dark_web', when(upper(col('card_on_dark_web')) == 'YES', True)
        .when(upper(col('card_on_dark_web')) == 'NO', False)
        .otherwise(None))\
    .filter(col('card_id').isNotNull())\
    .filter(col('client_id').isNotNull())\
    .drop('cvv','card_number')\
    .dropDuplicates(['card_id','client_id'])

In [25]:
cards_processed.show(30, truncate=False)

+-------+---------+----------+---------------+----------+--------+----------------+------------+--------------+---------------------+----------------+----------+
|card_id|client_id|card_brand|card_type      |expires   |has_chip|num_cards_issued|credit_limit|acct_open_date|year_pin_last_changed|card_on_dark_web|is_expired|
+-------+---------+----------+---------------+----------+--------+----------------+------------+--------------+---------------------+----------------+----------+
|0      |1362     |AMEX      |CREDIT         |2024-04-30|true    |2               |33900.00    |1991-01-01    |2014                 |false           |true      |
|1      |550      |MASTERCARD|CREDIT         |2024-06-30|true    |1               |11600.00    |1994-01-01    |2013                 |false           |true      |
|2      |556      |MASTERCARD|DEBIT          |2021-09-30|true    |1               |19948.00    |1995-01-01    |2011                 |false           |true      |
|3      |1937     |VISA     

In [30]:
cards_processed\
    .write\
    .parquet('data/processed/cards')

In [31]:
transactions_processed\
    .write\
    .parquet('data/processed/transactions')

In [32]:
users_processed\
    .write\
    .parquet('data/processed/users')

In [33]:
transactions_df = spark.read.parquet('data/processed/transactions')
users_df = spark.read.parquet('data/processed/users')
cards_df = spark.read.parquet('data/processed/cards')

transactions_df.createOrReplaceTempView('transactions')
users_df.createOrReplaceTempView('users')
cards_df.createOrReplaceTempView('cards')

In [39]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customer_financial_metrics AS
SELECT
    u.client_id,
    COUNT(DISTINCT t.transaction_id) AS total_transactions,
    SUM(t.amount) AS total_spent,
    AVG(t.amount) AS avg_transaction_value,
    u.current_age,
    u.gender,
    u.credit_score,
    SUM(c.credit_limit) AS total_credit_limit,
    COUNT(DISTINCT c.card_id) AS num_cards,
    MAX(c.card_on_dark_web) AS card_on_dark_web,
    u.yearly_income,
    u.per_capita_income,
    u.total_debt,
    MIN(t.transaction_date) AS first_transaction_date,
    MAX(t.transaction_date) AS last_transaction_date
FROM users u
LEFT JOIN transactions t
    ON u.client_id = t.client_id
LEFT JOIN cards c
    ON u.client_id = c.client_id
GROUP BY
    u.client_id,
    u.current_age,
    u.gender,
    u.credit_score,
    u.yearly_income,
    u.per_capita_income,
    u.total_debt
    """)

DataFrame[]

In [40]:
spark.sql("SELECT * FROM customer_financial_metrics").show()

[Stage 60:=======================>                                  (2 + 3) / 5]

+---------+------------------+-----------+---------------------+-----------+------+------------+------------------+---------+----------------+-------------+-----------------+----------+----------------------+---------------------+
|client_id|total_transactions|total_spent|avg_transaction_value|current_age|gender|credit_score|total_credit_limit|num_cards|card_on_dark_web|yearly_income|per_capita_income|total_debt|first_transaction_date|last_transaction_date|
+---------+------------------+-----------+---------------------+-----------+------+------------+------------------+---------+----------------+-------------+-----------------+----------+----------------------+---------------------+
|      117|                 0|       NULL|                 NULL|         20|  Male|         595|          20176.00|        3|           false|     30229.00|         14824.00|  43239.00|                  NULL|                 NULL|
|      124|              8811| 1151279.48|            32.665971|         31|

In [49]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW card_usage_risk_metrics AS
SELECT
    c.card_id,
    c.client_id,
    c.card_brand,
    c.card_type,
    c.is_expired,
    COUNT(t.transaction_id) AS transaction_count,
    SUM(t.amount) AS total_spend,
    SUM(CASE WHEN t.use_chip = true THEN 1 ELSE 0 END) AS chip_transactions,
    COUNT(t.transaction_id) AS total_transactions,
    SUM(CASE WHEN t.has_error = true THEN 1 ELSE 0 END) AS error_transactions,
    SUM(CASE WHEN t.has_error = true THEN 1 ELSE 0 END) * 1.0
        / COUNT(CASE WHEN t.transaction_id = 0 THEN 1 ELSE 0 END) AS error_rate
FROM cards c
LEFT JOIN transactions t
    ON c.card_id = t.card_id
GROUP BY
    c.card_id,
    c.client_id,
    c.card_brand,
    c.card_type,
    c.is_expired
    """)

DataFrame[]

In [50]:
spark.sql("SELECT * FROM card_usage_risk_metrics").show()

[Stage 90:>                                                         (0 + 1) / 1]

+-------+---------+----------+---------------+----------+-----------------+-----------+-----------------+------------------+------------------+------------------+
|card_id|client_id|card_brand|      card_type|is_expired|transaction_count|total_spend|chip_transactions|total_transactions|error_transactions|        error_rate|
+-------+---------+----------+---------------+----------+-----------------+-----------+-----------------+------------------+------------------+------------------+
|      1|      550|MASTERCARD|         CREDIT|      true|             2841|  171596.06|             1665|              2841|                44|0.0154875043998592|
|     12|     1156|      VISA|         CREDIT|      true|             2419|  318490.50|             2419|              2419|                37|0.0152955766845804|
|     13|      858|MASTERCARD|         CREDIT|      true|             3477|  250025.36|             1986|              3477|               109|0.0313488639631867|
|     22|     1234|   

In [51]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW merchant_geography_metrics AS
SELECT
    merchant_city,
    merchant_state,
    mcc,
    COUNT(transaction_id) AS transaction_count,
    SUM(amount) AS total_spend,
    SUM(CASE WHEN has_error = true THEN 1 ELSE 0 END) AS error_count,
    SUM(CASE WHEN has_error = true THEN 1 ELSE 0 END) * 1.0
        / COUNT(transaction_id ) AS error_rate
FROM transactions
GROUP BY
    merchant_city,
    merchant_state,
    mcc
    """)

DataFrame[]

In [52]:
spark.sql('SELECT * FROM merchant_geography_metrics').show()

[Stage 91:==============================================>           (4 + 1) / 5]

+-------------+--------------+----+-----------------+-----------+-----------+------------------+
|merchant_city|merchant_state| mcc|transaction_count|total_spend|error_count|        error_rate|
+-------------+--------------+----+-----------------+-----------+-----------+------------------+
|         ERIE|            PA|7538|              116|    7849.29|          3|0.0258620689655172|
|MORRIS PLAINS|            NJ|7538|             1881|  108375.89|         17|0.0090377458798511|
|      NEWTOWN|            PA|4829|             9753|  945860.00|        350|0.0358863939300728|
|     FRANKLIN|            TX|5651|              422|   18849.43|          4|0.0094786729857820|
| SUNLAND PARK|            NM|7538|              499|   23637.34|         18|0.0360721442885772|
|     CAMPBELL|            TX|5541|             5254|  313373.00|         95|0.0180814617434336|
|      MILFORD|            ME|7538|               72|    5485.23|          1|0.0138888888888889|
|    FAIRFIELD|            OH|

In [53]:
spark.table('customer_financial_metrics')\
    .write\
    .parquet('data/curated/customer_financial_metrics')

In [54]:
spark.table('card_usage_risk_metrics')\
    .write\
    .parquet('data/curated/card_usage_risk_metrics')

In [55]:
spark.table('merchant_geography_metrics')\
    .write\
    .parquet('data/curated/merchant_geography_metrics')

In [58]:
city_transaction_metrics = transactions_processed\
        .filter(col('merchant_city').isNotNull())\
        .groupBy('merchant_city')\
        .agg(
            count('transaction_id').alias('total_transactions'),
            sum('amount').alias('total_spent'),
            avg('amount').alias('avg_transaction_value'))\
        .orderBy(col('total_transactions').desc())

city_transaction_metrics.show(10, truncate=False)


[Stage 119:=============================================>           (4 + 1) / 5]

+-------------+------------------+-----------+---------------------+
|merchant_city|total_transactions|total_spent|avg_transaction_value|
+-------------+------------------+-----------+---------------------+
|ONLINE       |1563696           |94617656.25|60.508984            |
|HOUSTON      |146883            |8425397.96 |57.361287            |
|MIAMI        |87384             |4232140.76 |48.431529            |
|BROOKLYN     |84020             |3487321.48 |41.505850            |
|LOS ANGELES  |81997             |3754746.39 |45.791265            |
|CHICAGO      |72537             |3141126.71 |43.303786            |
|DALLAS       |71862             |3644876.97 |50.720506            |
|LOUISVILLE   |66082             |3570543.85 |54.032019            |
|PHILADELPHIA |61411             |3507894.61 |57.121601            |
|SAN ANTONIO  |59262             |2991352.98 |50.476747            |
+-------------+------------------+-----------+---------------------+
only showing top 10 rows


In [64]:
card_type_metrics = cards_processed\
        .filter(col('card_type').isNotNull())\
        .groupBy('card_type')\
        .agg(
            count('credit_limit').alias('total_card_value'),
            sum('credit_limit').alias('total_credit_limit'),
            avg('credit_limit').alias('avg_credit_value'))\
        .orderBy(col('total_card_value').desc())

card_type_metrics.show(truncate=False)


+---------------+----------------+------------------+----------------+
|card_type      |total_card_value|total_credit_limit|avg_credit_value|
+---------------+----------------+------------------+----------------+
|DEBIT          |3511            |65156747.00       |18557.888636    |
|CREDIT         |2057            |22985700.00       |11174.380165    |
|DEBIT (PREPAID)|578             |37251.00          |64.448097       |
+---------------+----------------+------------------+----------------+



In [67]:
credit_score_metrics = users_processed\
        .filter(col('current_age').isNotNull())\
        .groupBy('current_age')\
        .agg(
            sum('credit_score').alias('total_credit_score'),
            avg('credit_score').alias('avg_credit_score'))\
        .orderBy(col('total_credit_score'))
            
credit_score_metrics.show(10, truncate=False)

+-----------+------------------+-----------------+
|current_age|total_credit_score|avg_credit_score |
+-----------+------------------+-----------------+
|99         |709               |709.0            |
|93         |770               |770.0            |
|94         |1472              |736.0            |
|98         |1546              |773.0            |
|89         |2199              |733.0            |
|92         |2873              |718.25           |
|91         |3135              |783.75           |
|90         |4121              |686.8333333333334|
|72         |4389              |731.5            |
|87         |4962              |708.8571428571429|
+-----------+------------------+-----------------+
only showing top 10 rows


In [75]:
c = cards_processed.alias('c')
t = transactions_processed.alias('t')

card_type_metrics = c.join(t, c.client_id == t.client_id, 'Inner')\
        .select(col('c.client_id').alias('client_id'),col('c.card_type'), col('t.amount'))\
        .groupBy('card_type')\
        .agg(
            count('*').alias('transaction_count'),
            sum('t.amount').alias('total_transaction_amount'),
            avg('t.amount').alias('avg_transaction_value'))\
        .filter(col('c.card_type').isNotNull())\
        .orderBy(col('total_transaction_amount').desc())

card_type_metrics.show(10, truncate=False)

[Stage 164:==============>                                          (1 + 3) / 4]

+---------------+-----------------+------------------------+---------------------+
|card_type      |transaction_count|total_transaction_amount|avg_transaction_value|
+---------------+-----------------+------------------------+---------------------+
|DEBIT          |29897625         |1574629897.48           |52.667391            |
|CREDIT         |16662966         |879875672.01            |52.804265            |
|DEBIT (PREPAID)|4503164          |234407264.20            |52.053903            |
+---------------+-----------------+------------------------+---------------------+

